# Test dữ liệu MVP — VN30F1M

Đọc toàn bộ CSV 1 phút đã tải về, xem lịch sử thực tế và thử gộp thành 5 phút / ngày. Chọn kernel `.venv/bin/python`, chạy các cell từ trên xuống. Không gọi API, không xuất CSV/Parquet.

`df_raw` giữ dữ liệu CSV gốc; `df` là bảng 1m sau chuẩn hóa; `df_5m` và `df_1d` là dữ liệu tổng hợp. MVP này chỉ chứa VN30F1M, không phải danh sách cổ phiếu.


In [1]:
from pathlib import Path
import os
import sys

import pandas as pd
import numpy as np
from dotenv import load_dotenv
from IPython.display import display

PROJECT_ROOT = next(
    (p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (p / "ingest" / "market" / "pipeline.py").is_file()), None
)
if PROJECT_ROOT is None:
    raise RuntimeError("Mở notebook từ thư mục project.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
load_dotenv(PROJECT_ROOT / ".env", override=True)

from transform.silver.normalization import normalize_ohlcv, resample_ohlcv

# Để None để lấy đường dẫn từ MVP_CSV_PATH trong .env hoặc mặc định bên dưới.
CSV_PATH = None
csv_path = Path(CSV_PATH or os.getenv("MVP_CSV_PATH") or "input/mvp/vn30f1m_1m.csv").expanduser()
if not csv_path.is_absolute():
    csv_path = PROJECT_ROOT / csv_path
if not csv_path.is_file():
    raise FileNotFoundError("Chưa có CSV MVP. Đặt MVP_CSV_PATH trong .env hoặc sửa CSV_PATH ở cell này.")
print("CSV:", csv_path.name)


CSV: vn30f1m_1m.csv


In [2]:
df_raw = pd.read_csv(csv_path)
print("Raw rows:", len(df_raw))
print("Columns:", list(df_raw.columns))
display(df_raw.head())
display(df_raw.tail())
display(df_raw.isna().sum().rename("null_count").to_frame())


Raw rows: 352195
Columns: ['trading_date', 'bar_time', 'tickersymbol', 'contract_month', 'open', 'high', 'low', 'close', 'trade_count', 'matched_volume', 'first_trade_time', 'last_trade_time']


,trading_date,bar_time,tickersymbol,contract_month,open,high,low,close,trade_count,matched_volume,first_trade_time,last_trade_time
0,2021-01-15,2021-01-15 10:29:00,VN30F2101,2021-01-01,1193.8,1193.8,1193.8,1193.8,1,5.0,2021-01-15 10:29:55.249470,2021-01-15 10:29:55.249470
1,2021-01-15,2021-01-15 10:30:00,VN30F2101,2021-01-01,1194.0,1194.0,1193.8,1194.0,9,20.0,2021-01-15 10:30:01.751771,2021-01-15 10:30:51.838172
2,2021-01-15,2021-01-15 10:31:00,VN30F2101,2021-01-01,1194.1,1194.2,1194.1,1194.2,4,13.0,2021-01-15 10:31:39.810469,2021-01-15 10:31:54.627576
3,2021-01-15,2021-01-15 10:32:00,VN30F2101,2021-01-01,1194.1,1194.1,1193.9,1194.0,5,6.0,2021-01-15 10:32:01.092554,2021-01-15 10:32:28.071678
4,2021-01-15,2021-01-15 10:33:00,VN30F2101,2021-01-01,1193.9,1193.9,1193.7,1193.7,3,7.0,2021-01-15 10:33:07.148797,2021-01-15 10:33:21.834399


,trading_date,bar_time,tickersymbol,contract_month,open,high,low,close,trade_count,matched_volume,first_trade_time,last_trade_time
352190,2026-06-23,2026-06-23 13:11:00,VN30F2607,2026-07-01,2008.0,2009.9,2008.0,2009.8,187,446.0,2026-06-23 13:11:00.633081,2026-06-23 13:11:59.959565
352191,2026-06-23,2026-06-23 13:12:00,VN30F2607,2026-07-01,2009.9,2011.0,2007.1,2009.3,205,594.0,2026-06-23 13:12:00.086009,2026-06-23 13:12:59.850499
352192,2026-06-23,2026-06-23 13:13:00,VN30F2607,2026-07-01,2009.1,2011.5,2008.7,2011.2,98,290.0,2026-06-23 13:13:00.097594,2026-06-23 13:13:59.461594
352193,2026-06-23,2026-06-23 13:14:00,VN30F2607,2026-07-01,2011.7,2012.0,2008.5,2008.7,165,272.0,2026-06-23 13:14:00.028979,2026-06-23 13:14:59.652570
352194,2026-06-23,2026-06-23 13:15:00,VN30F2607,2026-07-01,2009.1,2010.4,2008.6,2008.7,95,345.0,2026-06-23 13:15:00.097689,2026-06-23 13:15:34.386344


,null_count
trading_date,0
bar_time,0
tickersymbol,0
contract_month,0
open,0
high,0
low,0
close,0
trade_count,0
matched_volume,24969


In [3]:
df = normalize_ohlcv(df_raw)
df["symbol"] = "VN30F1M"
df["source"] = "mvp"
df["asset"] = "futures"
df["interval"] = "1m"
print("Normalized rows:", len(df))
print("Rows removed by normalization:", len(df_raw) - len(df))
print("Lưu ý: helper chuẩn hóa có thể bỏ dòng lỗi thời gian/OHLC và điền volume thiếu bằng 0; xem df_raw để đối chiếu.")
display(df.head())
display(df.tail())


Normalized rows: 352195
Rows removed by normalization: 0
Lưu ý: helper chuẩn hóa có thể bỏ dòng lỗi thời gian/OHLC và điền volume thiếu bằng 0; xem df_raw để đối chiếu.


,time,open,high,low,close,volume,trade_count,source_contract,contract_month,symbol,source,asset,interval
0,2021-01-15 10:29:00+07:00,1193.8,1193.8,1193.8,1193.8,5.0,1,VN30F2101,2021-01-01,VN30F1M,mvp,futures,1m
1,2021-01-15 10:30:00+07:00,1194.0,1194.0,1193.8,1194.0,20.0,9,VN30F2101,2021-01-01,VN30F1M,mvp,futures,1m
2,2021-01-15 10:31:00+07:00,1194.1,1194.2,1194.1,1194.2,13.0,4,VN30F2101,2021-01-01,VN30F1M,mvp,futures,1m
3,2021-01-15 10:32:00+07:00,1194.1,1194.1,1193.9,1194.0,6.0,5,VN30F2101,2021-01-01,VN30F1M,mvp,futures,1m
4,2021-01-15 10:33:00+07:00,1193.9,1193.9,1193.7,1193.7,7.0,3,VN30F2101,2021-01-01,VN30F1M,mvp,futures,1m


,time,open,high,low,close,volume,trade_count,source_contract,contract_month,symbol,source,asset,interval
352190,2026-06-23 13:11:00+07:00,2008.0,2009.9,2008.0,2009.8,446.0,187,VN30F2607,2026-07-01,VN30F1M,mvp,futures,1m
352191,2026-06-23 13:12:00+07:00,2009.9,2011.0,2007.1,2009.3,594.0,205,VN30F2607,2026-07-01,VN30F1M,mvp,futures,1m
352192,2026-06-23 13:13:00+07:00,2009.1,2011.5,2008.7,2011.2,290.0,98,VN30F2607,2026-07-01,VN30F1M,mvp,futures,1m
352193,2026-06-23 13:14:00+07:00,2011.7,2012.0,2008.5,2008.7,272.0,165,VN30F2607,2026-07-01,VN30F1M,mvp,futures,1m
352194,2026-06-23 13:15:00+07:00,2009.1,2010.4,2008.6,2008.7,345.0,95,VN30F2607,2026-07-01,VN30F1M,mvp,futures,1m


In [4]:
required = ["time", "open", "high", "low", "close", "volume"]
prices = df[["open", "high", "low", "close"]]
invalid_range = (
    (df["high"] < df["low"])
    | (df["open"] < df["low"]) | (df["open"] > df["high"])
    | (df["close"] < df["low"]) | (df["close"] > df["high"])
)
duplicate = df.duplicated(["symbol", "time"], keep=False)
non_finite = ~np.isfinite(df[["open", "high", "low", "close", "volume"]].to_numpy(dtype=float)).all(axis=1)
display(pd.DataFrame([{
    "rows": len(df),
    "null_required_values": int(df[required].isna().sum().sum()),
    "duplicate_rows_all_occurrences": int(duplicate.sum()),
    "invalid_ohlc_ranges": int(invalid_range.sum()),
    "non_positive_prices": int(prices.le(0).any(axis=1).sum()),
    "negative_volume": int(df["volume"].lt(0).sum()),
    "non_finite_rows": int(non_finite.sum()),
}]))
issues = df.loc[invalid_range | duplicate | non_finite | prices.le(0).any(axis=1) | df["volume"].lt(0)]
display(issues.head(20))


,rows,null_required_values,duplicate_rows_all_occurrences,invalid_ohlc_ranges,non_positive_prices,negative_volume,non_finite_rows
0,352195,0,0,0,0,0,0


,time,open,high,low,close,volume,trade_count,source_contract,contract_month,symbol,source,asset,interval


## Gộp 1m thành 5m và 1D

Bảng dưới dùng quy tắc resample hiện tại của project. Nếu timestamp trùng thì dừng để tránh cộng volume hai lần. Cờ `is_complete_window` chỉ đếm số dòng, chưa xác nhận lịch phiên hoặc chất lượng từng phút. Chưa điều chỉnh chênh lệch giá khi chuyển hợp đồng tương lai.


In [5]:
if df.duplicated(["symbol", "time"]).any():
    raise ValueError("Có timestamp trùng; kiểm tra issues trước khi gộp nến.")
df_5m = resample_ohlcv(df, "5m").assign(symbol="VN30F1M", interval="5m")
df_1d = resample_ohlcv(df, "1D").assign(symbol="VN30F1M", interval="1D")
coverage = pd.DataFrame([
    {
        "symbol": "VN30F1M", "interval": interval, "rows": len(frame),
        "first_time": frame["time"].min(), "last_time": frame["time"].max(),
        "observed_days": frame["time"].dt.date.nunique(),
    }
    for interval, frame in [("1m", df), ("5m", df_5m), ("1D", df_1d)]
])
display(coverage)
display(df_5m.head())
display(df_1d.head())
print("5m windows with fewer than 5 source rows:", int((~df_5m["is_complete_window"]).sum()))


5m windows with fewer than 5 source rows: 6756


,symbol,interval,rows,first_time,last_time,observed_days
0,VN30F1M,1m,352195,2021-01-15 10:29:00+07:00,2026-06-23 13:15:00+07:00,1337
1,VN30F1M,5m,73771,2021-01-15 10:25:00+07:00,2026-06-23 13:15:00+07:00,1337
2,VN30F1M,1D,1337,2021-01-15 00:00:00+07:00,2026-06-23 00:00:00+07:00,1337


,time,open,high,low,close,volume,source_rows,trade_count,source_contract,contract_month,is_complete_window,symbol,interval
0,2021-01-15 10:25:00+07:00,1193.8,1193.8,1193.8,1193.8,5.0,1,1,VN30F2101,2021-01-01,False,VN30F1M,5m
1,2021-01-15 10:30:00+07:00,1194.0,1194.2,1193.5,1193.5,65.0,5,24,VN30F2101,2021-01-01,True,VN30F1M,5m
2,2021-01-15 10:35:00+07:00,1193.6,1193.7,1193.5,1193.6,1.0,2,6,VN30F2101,2021-01-01,False,VN30F1M,5m
3,2021-02-08 08:45:00+07:00,1159.4,1159.4,1135.0,1135.0,0.0,4,17,VN30F2102,2021-02-01,False,VN30F1M,5m
4,2021-02-08 08:50:00+07:00,1134.0,1136.0,1134.0,1135.8,0.0,4,16,VN30F2102,2021-02-01,False,VN30F1M,5m


,time,open,high,low,close,volume,source_rows,trade_count,source_contract,contract_month,symbol,interval
0,2021-01-15 00:00:00+07:00,1193.8,1194.2,1193.5,1193.6,71.0,8,31,VN30F2101,2021-01-01,VN30F1M,1D
1,2021-02-08 00:00:00+07:00,1159.4,1159.4,1086.2,1096.0,7190.0,164,2093,VN30F2102,2021-02-01,VN30F1M,1D
2,2021-02-09 00:00:00+07:00,1123.0,1141.9,1095.0,1130.3,10199.0,264,3425,VN30F2102,2021-02-01,VN30F1M,1D
3,2021-02-17 00:00:00+07:00,1164.4,1180.0,1161.5,1176.6,2767.0,106,1300,VN30F2102,2021-02-01,VN30F1M,1D
4,2021-02-18 00:00:00+07:00,1178.8,1193.8,1168.3,1187.3,7605.0,263,3019,VN30F2102,2021-02-01,VN30F1M,1D


In [6]:
# Số dòng theo năm giúp thấy phân bố lịch sử; không chứng minh đầy đủ mọi phiên.
if not df.empty:
    yearly_coverage = df.groupby(df["time"].dt.year).agg(
        rows=("time", "size"), first_time=("time", "min"), last_time=("time", "max")
    )
    display(yearly_coverage)
# Xem tùy ý: df, df_raw, df_5m, df_1d


,rows,first_time,last_time
time,,,
2021,58906,2021-01-15 10:29:00+07:00,2021-12-31 14:46:00+07:00
2022,65757,2022-01-04 08:45:00+07:00,2022-12-30 14:45:00+07:00
2023,65651,2023-01-03 08:45:00+07:00,2023-12-29 14:45:00+07:00
2024,66461,2024-01-02 08:45:00+07:00,2024-12-31 14:45:00+07:00
2025,65422,2025-01-02 08:45:00+07:00,2025-12-31 14:44:00+07:00
2026,29998,2026-01-05 08:45:00+07:00,2026-06-23 13:15:00+07:00
